# Transient transport: UFL spatial forms and explicit backward Euler

Write the mass, conservative transport, local interface and global balance
before choosing assembly or solvers. Time stepping updates only source loads
while the generic offline system owns unchanged operator factors.

This notebook has no ready-made transport solver. The time loop follows the
backward-Euler mathematical equations. Run with the locked `introduction`
profile; the actual native UFL path always executes.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/d4d2a3374809c152fbd317ca00e3a43f8aacbdcd5c16ff8cf8d87217d9386cd2/transient_variational-companion.zip"
COMPANION_SHA256 = "d4d2a3374809c152fbd317ca00e3a43f8aacbdcd5c16ff8cf8d87217d9386cd2"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/transient_variational.ipynb", directory=ROOT)


In [ ]:
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("PYMHM_WORKSPACE", Path.cwd())).resolve()
if not (ROOT / "examples").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "examples").is_dir()), ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "build" / "native-teaching"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Workspace:", ROOT)


## 1. Manufactured concentration and physical flux

Use capacity $\rho=1$, diffusion $K=1$, reaction $\sigma=1/2$ and constant
transport velocity $\beta=(1/4,-1/2)$. On the unit square,

$$
\begin{aligned}
\partial_t c+\nabla\cdot(-\nabla c+\beta c)+\tfrac12 c&=f,\\
c(x,y,t)&=e^{-t}\sin(\pi x)\sin(\pi y),\\
f(x,y,t)&=e^{-t}[(2\pi^2-\tfrac12)\phi+\beta\cdot\nabla\phi],\\
\phi(x,y)&=\sin(\pi x)\sin(\pi y).
\end{aligned}
$$

The exterior concentration is zero. The **physical transport flux** is
$j=-\nabla c+\beta c$; the local half-advection multiplier is
$\lambda_K=(-\nabla c+\beta c/2)\cdot n_K$.
Initial values are independently projected in the local capacity-weighted L2
mass, rather than assigned by a discrete manufactured operator.


In [ ]:
import numpy as np
import ufl
from pymhm import (
    TriangleMesh,FaceSpace,SkeletonSpace,MeshHierarchy,LocalContext,
    Equation,columns,compile_form,bind_interface,bind_problem,assemble,
)
from pymhm.core.online import OfflineMultiscaleSystem
from pymhm.linalg.linear import factorize,solve_linear
from pymhm.fem.scalar.operators import triangle_quadrature

def exact_concentration(points,time):
    """Return independently manufactured physical concentration."""
    return np.exp(-time)*np.sin(np.pi*points[:,0])*np.sin(np.pi*points[:,1])

macro=TriangleMesh.unit_square(2)
local_refinement=16
skeleton=SkeletonSpace(macro,tuple(FaceSpace.uniform(1,local_refinement // 2) for _ in macro.faces))
hierarchy=MeshHierarchy(macro,tuple(macro.submesh(i,local_refinement) for i in range(len(macro.cells))))
interface=bind_interface(skeleton,convention="normal")


## 2. Declare the backward-Euler local equations

For each time increment $\Delta t$, write

$$
\begin{aligned}
\tfrac1{\Delta t}(c^{n+1},v)_K+a_K(c^{n+1},v)
 +\langle\lambda_K^{n+1},v\rangle&=(f^{n+1},v)_K
 +\tfrac1{\Delta t}(c^n,v)_K,\\
a_K(c,v)&=(\nabla c,\nabla v)_K+\tfrac12(c,v)_K\\
&\quad+\tfrac12[(\beta\cdot\nabla c,v)_K-(c,\beta\cdot\nabla v)_K].
\end{aligned}
$$

The source and previous-state mass use the same Galerkin test. A SUPG variant
would require consistent Petrov terms in **both** source and old-state mass.
The positive mass removes the constant nullspace, so the constant below is a
retained coarse mode, not `kernel`. The piecewise P1 trace uses half as many face segments as the fine local P2 mesh.
This enriched local trace lifting avoids redundant multipliers; temporal errors
must still be measured above its fixed spatial approximation floor.


In [ ]:
def local_time_equations(local: LocalContext,dt: float):
    """Write Q-independent triangular P2 mass, skew transport and interface forms."""
    binding=local.native_space(degree=2)
    V=binding.space
    c,v=ufl.TrialFunction(V),ufl.TestFunction(V)
    x=ufl.SpatialCoordinate(binding.mesh)
    beta=ufl.as_vector((0.25,-0.5))
    phi=ufl.sin(np.pi*x[0])*ufl.sin(np.pi*x[1])
    grad_phi=np.pi*ufl.as_vector((ufl.cos(np.pi*x[0])*ufl.sin(np.pi*x[1]),
                                ufl.sin(np.pi*x[0])*ufl.cos(np.pi*x[1])))
    force_profile=(2*np.pi**2-0.5)*phi+ufl.dot(beta,grad_phi)
    dx=ufl.Measure("dx",domain=binding.mesh,metadata={"quadrature_degree":12})
    mass=compile_form(c*v*dx)
    initial_load=compile_form(phi*v*dx)
    initial=solve_linear(mass,initial_load)
    source_profile=compile_form(force_profile*v*dx)
    a=((1/dt+0.5)*c*v+ufl.inner(ufl.grad(c),ufl.grad(v))
       +0.5*(ufl.dot(beta,ufl.grad(c))*v-c*ufl.dot(beta,ufl.grad(v))))*dx
    L=np.exp(-dt)*source_profile+mass@initial/dt
    b=local.trace_pairings(lambda phi,ds:phi*v*ds)
    balance=local.trace_pairings(lambda phi,ds:-phi*c*ds,axis="rows")
    local.field("concentration",binding)
    return local.equations(
        a=a,L=L,b=b,c=balance,
        coarse_basis=np.ones((binding.size,1)),moments=columns(v*dx),
        metadata={"mass":mass,"initial":initial,"source_profile":source_profile},
    )


## 3. Global equation, assembly and unchanged operator reuse

Zero exterior concentration means the additional global load is zero. Local
`c` pairings impose continuity and those boundary moments. No pressure-style
gauge is required for this scalar time-step operator.

A fixed $\Delta t$ permits one local/global factor preparation. Changing the
step size changes the operator. The time loop writes the source update and
calls the existing numerical owners; it implements no alternate elimination or
factorization algorithm.


In [ ]:
from functools import partial

def trajectory(dt,final_time=1.0):
    """Advance the explicitly declared backward-Euler equations at a fixed step."""
    steps=int(round(final_time/dt))
    if not np.isclose(steps*dt,final_time): raise ValueError("dt must divide final time")
    problem=bind_problem(hierarchy,interface,partial(local_time_equations,dt=dt),
                         global_equation=Equation(0,0),retained=1)
    system=assemble(problem)
    records=system.local_metadata
    previous=tuple(record["initial"] for record in records)
    maximum_residual=0.0
    with OfflineMultiscaleSystem(system) as prepared, factorize(system.matrix) as global_factor:
        for step in range(1,steps+1):
            time=step*dt
            loads=tuple(np.exp(-time)*record["source_profile"]+record["mass"]@old/dt
                        for record,old in zip(records,previous,strict=True))
            updated=prepared.with_loads(loads)
            solution=updated.solve(factorization=global_factor)
            maximum_residual=max(maximum_residual,float(solution.raw_residual))
            previous=solution.fields
    return solution,maximum_residual

solution,maximum_residual=trajectory(0.05)
concentration=solution.field("concentration")
print({"maximum_original_equation_residual":maximum_residual,"steps":20,"operator_builds":1})


## 4. Integrate the physical concentration error

Integrate independently on each fine triangle, with physical area weights and
polynomial-exact quadrature sufficient for the represented field. Keep the
same spatial spaces in the time-refinement experiment. The analytical solution
is available, so physical field errors have no numerical-reference ambiguity.


In [ ]:
def concentration_error(solution,final_time=1.0,order=8):
    """Integrate the concentration L2 error independently on every fine cell."""
    bary,weights=triangle_quadrature(order)
    squared=0.0
    for field in solution.field("concentration"):
        fine=field.mesh
        points=np.einsum("qi,tia->tqa",bary,fine.points[fine.cells])
        error=field.evaluate(points.reshape(-1,2)).reshape(points.shape[:2])-exact_concentration(points.reshape(-1,2),final_time).reshape(points.shape[:2])
        squared+=float(np.sum(fine.areas[:,None]*weights*error**2))
    return float(np.sqrt(squared))

print("Concentration L2 error:",concentration_error(solution))


## 5. Verify the independent conforming Galerkin baseline

Use the same capacity, diffusion, conservative advection, reaction, source and
boundary data on globally conforming P2 meshes. Each reference uses backward
Euler with the smallest tutorial time step. Three independent mesh resolutions
measure the reference's own spatial refinement difference; this remains a
numerical baseline, while the analytical concentration supplies the exact norm.
The fine reference is used for comparison and is not the MHM macro mesh.


In [ ]:
import basix
import basix.ufl
from pymhm import MultiscaleProblem,solve
from pymhm.backends.spaces import bind_space
from pymhm.core.multiscale import with_global_load
from pymhm.postprocessing.fields import DiscreteField,FieldDefinition

def classical_trajectory(n,dt=0.00625,final_time=1.0):
    """Advance an independently assembled globally conforming P2 Galerkin equation."""
    fine=TriangleMesh.unit_square(n)
    element=basix.ufl.element("Lagrange","triangle",2,lagrange_variant=basix.LagrangeVariant.equispaced)
    binding=bind_space(fine,element)
    try:
        V=binding.space
        c,v=ufl.TrialFunction(V),ufl.TestFunction(V)
        x=ufl.SpatialCoordinate(binding.mesh)
        beta=ufl.as_vector((0.25,-0.5))
        phi=ufl.sin(np.pi*x[0])*ufl.sin(np.pi*x[1])
        grad_phi=np.pi*ufl.as_vector((ufl.cos(np.pi*x[0])*ufl.sin(np.pi*x[1]),
                                    ufl.sin(np.pi*x[0])*ufl.cos(np.pi*x[1])))
        source=(2*np.pi**2-0.5)*phi+ufl.dot(beta,grad_phi)
        dx=ufl.Measure("dx",domain=binding.mesh,metadata={"quadrature_degree":12})
        mass=compile_form(c*v*dx)
        initial_load=compile_form(phi*v*dx)
        source_load=compile_form(source*v*dx)
        a=((1/dt+0.5)*c*v+ufl.inner(ufl.grad(c),ufl.grad(v))
           +0.5*(ufl.dot(beta,ufl.grad(c))*v-c*ufl.dot(beta,ufl.grad(v))))*dx
        coordinates=V.tabulate_dof_coordinates()[:,:2]
        boundary=np.flatnonzero(np.any(np.isclose(coordinates,0)|np.isclose(coordinates,1),axis=1))
        fixed={int(i):0.0 for i in boundary}
        initial_problem=MultiscaleProblem.from_global(Equation(mass,initial_load),binding.size,fixed=fixed)
        previous=solve(assemble(initial_problem)).trace
        definition=MultiscaleProblem.from_global(Equation(a,source_load),binding.size,fixed=fixed)
        system=assemble(definition)
        free=np.setdiff1d(np.arange(binding.size),boundary)
        steps=int(round(final_time/dt))
        with factorize(system.matrix[free][:,free]) as factor:
            for step in range(1,steps+1):
                load=np.exp(-step*dt)*source_load+mass@previous/dt
                previous=solve(with_global_load(system,load),factorization=factor).trace
        return DiscreteField(FieldDefinition("concentration",descriptor=binding.descriptor()),previous)
    finally:
        binding.close()

def field_distance(first,second,order=8):
    """Measure a physical concentration difference on the finer reference geometry."""
    bary,weights=triangle_quadrature(order)
    fine=second.mesh
    points=np.einsum("qi,tia->tqa",bary,fine.points[fine.cells])
    flattened=points.reshape(-1,2)
    difference=(first.evaluate(flattened)-second.evaluate(flattened)).reshape(points.shape[:2])
    return float(np.sqrt(np.sum(fine.areas[:,None]*weights*difference**2)))

references={n:classical_trajectory(n) for n in (16,32,64)}
reference=references[64]
reference_refinement={"16 to 32":field_distance(references[16],references[32]),
                      "32 to 64":field_distance(references[32],references[64])}
print("Classical P2 spatial refinement differences at the same dt:",reference_refinement)


## 5. Plot the final-time concentration

Draw analytical, reconstructed and error fields in separate panels. The original
macro mesh is visible in each panel and local incident values are preserved.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

def scalar_panel(ax, fields, exact, title, *, error=False):
    """Plot each macrocell independently, with original macro topology visible."""
    artists=[]
    for field in fields:
        fine=field.mesh
        values=field.evaluate(fine.points)
        if values.ndim==2: values=np.linalg.norm(values,axis=1)
        if exact is not None:
            truth=exact(fine.points)
            if truth.ndim==2: truth=np.linalg.norm(truth,axis=1)
            values=abs(values-truth) if error else truth
        artist=ax.tripcolor(fine.points[:,0],fine.points[:,1],fine.cells,values,shading="gouraud")
        artists.append(artist)
    lower=min(a.get_clim()[0] for a in artists);upper=max(a.get_clim()[1] for a in artists)
    for a in artists:a.set_clim(lower,upper)
    ax.add_collection(LineCollection(macro.points[macro.faces],colors="black",linewidths=0.65,alpha=0.55))
    ax.set(title=title,xlabel="x",ylabel="y",aspect="equal")
    return artists[-1]
figure,axes=plt.subplots(1,4,figsize=(15,4.2),layout="constrained")
for ax,title,truth,error in zip(axes,("Analytical\nconcentration","MHM\nconcentration","Classical P2\nconcentration","Absolute error\nconcentration"),
                              (lambda x:exact_concentration(x,1.0),None,None,lambda x:exact_concentration(x,1.0)),
                              (False,False,False,True),strict=True):
    chosen=(reference,) if title.startswith("Classical") else concentration
    artist=scalar_panel(ax,chosen,truth,title,error=error)
    figure.colorbar(artist,ax=ax,pad=0.025,shrink=0.85)
figure.savefig(OUTPUT/"transient-concentration-fields.png",dpi=180)
plt.show()


## 6. Temporal convergence with fixed spatial approximation

Backward Euler has first-order temporal error for this smooth solution.
Use several steps whose errors remain above the fixed spatial floor. Measure
the final consecutive rates rather than fitting one slope through coarse
meshes. Report original-equation checks at every time step.


In [ ]:
from examples.tutorial_convergence import plot_method_series
rows=[]
for dt in (0.1,0.05,0.025,0.0125,0.00625):
    state,residual=trajectory(dt)
    low=concentration_error(state,order=6)
    high=concentration_error(state,order=8)
    np.testing.assert_allclose(low,high,rtol=1e-8,atol=1e-12)
    rows.append({"dt":dt,"concentration_L2":high,"global_compatibility_relative":residual})
sizes=np.array([row["dt"] for row in rows])
errors=np.array([row["concentration_L2"] for row in rows])
rates=np.log(errors[:-1]/errors[1:])/np.log(sizes[:-1]/sizes[1:])
print(rows)
print("Observed backward-Euler orders:",rates)
figure=plot_method_series({"method":"transient-transport-time","sizes":sizes.tolist(),
    "errors":{"concentration L2":errors.tolist()},"rates":{"concentration L2":rates.tolist()},
    "expected":{"concentration L2":1.0},"refinement":"Time step Δt"})
for extension in ("png","svg","pdf"):
    figure.savefig(OUTPUT/f"transient-native-convergence.{extension}",dpi=180)
plt.show()

import json
import hashlib

# The public runner supplies the literal downloaded or checkout source path.
# A manually renamed notebook needs PYMHM_NOTEBOOK_SOURCE set explicitly.
source_file = Path(os.environ.get("PYMHM_NOTEBOOK_SOURCE", "transient_variational.ipynb"))
if not source_file.is_file() and "PYMHM_NOTEBOOK_SOURCE" not in os.environ:
    source_file = Path.cwd()/"notebooks/transport/transient_variational.ipynb"
source_bytes = source_file.read_bytes()
source_notebook = json.loads(source_bytes)
scientific_cells = [
    {"cell_type": cell["cell_type"],
     "source": "".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]}
    for cell in source_notebook["cells"]
    if "pymhm-companion-bootstrap" not in cell.get("metadata", {}).get("tags", [])
]
scientific_bytes = json.dumps(
    scientific_cells, ensure_ascii=False, sort_keys=True, separators=(",", ":")
).encode("utf-8")
study={
    "case":"native UFL MHM conservative transient transport",
    "operator":"capacity=1, diffusion=1, beta=(0.25,-0.5), reaction=0.5",
    "exact":"exp(-t)*sin(pi*x)*sin(pi*y)",
    "final_time":1.0,"macro_divisions":2,"macro_cells":len(macro.cells),
    "local_degree":2,"local_edge_subdivisions":local_refinement,
    "trace_degree":1,"face_segments":local_refinement//2,
    "time_integrator":"backward Euler","assembly_quadrature_degree":12,
    "error_quadrature_orders":[6,8],"rows":rows,"concentration_L2_rates":rates.tolist(),
    "classical_reference":{"family":"conforming P2 Galerkin","resolutions":[16,32,64],
                           "dt":0.00625,"spatial_L2_differences":reference_refinement},
    "source_notebook_sha256": hashlib.sha256(source_bytes).hexdigest(),
    "scientific_cells_sha256": hashlib.sha256(scientific_bytes).hexdigest(),
    "scientific_cells_hash_scope": (
        "UTF-8 JSON sequence of {cell_type, source}; sort_keys=True, "
        "ensure_ascii=False, separators=(comma,colon); excludes only cells "
        "tagged pymhm-companion-bootstrap, which install companion resources."
    ),
    "figures": {
        name: hashlib.sha256((OUTPUT/name).read_bytes()).hexdigest()
        for name in ("transient-concentration-fields.png", "transient-native-convergence.png",
                     "transient-native-convergence.svg", "transient-native-convergence.pdf")
    },
}
(OUTPUT/"transient-native-study.json").write_text(json.dumps(study,indent=2)+"\n")


## References and scope

- Harder, Paredes and Valentin (2015), *On a Multiscale Hybrid-Mixed Method for
  Advective-Reactive Dominated Problems with Heterogeneous Coefficients*.
  https://doi.org/10.1137/130938499
- Araya, Jaillet, Paredes and Valentin (2024), *Generalizing the multiscale
  hybrid-mixed method for reactive-advective-diffusive equations*.
  https://doi.org/10.1016/j.cma.2024.117089

The temporal order belongs to backward Euler. This smooth analytical experiment
is not a reproduction of a heterogeneous published transient geometry, a
maximum-principle claim, or a spatial convergence theorem.
